# Experiment 08 — External Paper Reproduction

## E7 — Temporal Order Sensitivity on ETTh1

### 1. Mục tiêu

Tái thực nghiệm và mở rộng thí nghiệm về khả năng khai thác thứ tự thời gian trong bài báo **"Are Transformers Effective for Time Series Forecasting?"** (Zeng et al., AAAI 2023).

Sử dụng ETTh1 làm bộ dữ liệu chung để:

- Tái thực nghiệm mô hình Linear và các phép biến đổi thứ tự chuỗi đầu vào.
- Đánh giá mức độ nhạy cảm với thứ tự thời gian của DeepKoopFormer.
- So sánh kết quả trong cùng điều kiện thực nghiệm.

### 2. Dataset — ETTh1

**Electricity Transformer Temperature (ETTh1)**

- Frequency: Hourly.
- Observations: 17,420 timestamps.
- Features: 7 numerical channels.
- Target variable: Oil Temperature (OT).
- Task: Multivariate Time Series Forecasting.

**Data preparation:**

- Kiểm tra tính toàn vẹn của dữ liệu.
- Chia Train/Validation/Test theo thứ tự thời gian.
- Chuẩn hóa dữ liệu bằng thống kê từ tập Train.
- Tạo Sliding Windows phục vụ huấn luyện và đánh giá.

### 3. Experimental Protocol

**Reference paper:**
- Zeng et al. (2023), AAAI.
- Reference experiment: Table 5 — Temporal Order Preservation.

**Input transformations:**

- Original — Giữ nguyên thứ tự thời gian.
- Shuffle — Hoán vị ngẫu nhiên thứ tự các timestamp.
- Half-Exchange — Hoán đổi hai nửa chuỗi đầu vào.

**Reference forecasting horizons:** 96, 192, 336, 720.

**Evaluation metrics:**
- Mean Squared Error (MSE).
- Mean Absolute Error (MAE).
- Relative Performance Degradation (%).

### 4. Quy trình thực nghiệm

1. `01_etth1_data.ipynb` — Chuẩn bị và kiểm tra dữ liệu ETTh1.
2. `02_linear_reproduction.ipynb` — Tái thực nghiệm mô hình Linear.
3. `03_temporal_order_test.ipynb` — Đánh giá ảnh hưởng của việc thay đổi thứ tự thời gian.
4. `04_deepkoopformer_comparison.ipynb` — Mở rộng thí nghiệm cho DeepKoopFormer.

### 5. Kết quả đầu ra

- Bộ dữ liệu ETTh1 đã tiền xử lý.
- Thông tin chia tập và tham

LOAD & VALIDATE ETTh1

In [2]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


# 1. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "datasets" / "ETTh1" / "ETTh1.csv").is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate datasets/ETTh1/ETTh1.csv"
    )

# 2. Directories
EXTERNAL_DIR = PROJECT_ROOT / "experiments" / "08_external_paper"

DATA_DIR = PROJECT_ROOT / "datasets" / "ETTh1"
DATA_PATH = DATA_DIR / "ETTh1.csv"

RESULTS_DIR = EXTERNAL_DIR / "results" / "etth1_data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Expected ETTh1 schema
FEATURE_COLUMNS = [
    "HUFL", "HULL",
    "MUFL", "MULL",
    "LUFL", "LULL",
    "OT"
]

EXPECTED_COLUMNS = ["date"] + FEATURE_COLUMNS
EXPECTED_ROWS = 17420

# 3. Load raw CSV
ETTH1_DF = pd.read_csv(DATA_PATH)

assert ETTH1_DF.columns.tolist() == EXPECTED_COLUMNS, (
    f"Unexpected columns: {ETTH1_DF.columns.tolist()}"
)

assert ETTH1_DF.shape == (EXPECTED_ROWS, 8), (
    f"Unexpected shape: {ETTH1_DF.shape}"
)

# 4. Parse timestamps
ETTH1_DF["date"] = pd.to_datetime(
    ETTH1_DF["date"],
    errors="raise"
)

assert ETTH1_DF["date"].notna().all()
assert ETTH1_DF["date"].is_unique
assert ETTH1_DF["date"].is_monotonic_increasing

# Check hourly frequency
time_diffs = ETTH1_DF["date"].diff().dropna()

assert (
    time_diffs == pd.Timedelta(hours=1)
).all(), "Missing or irregular hourly timestamps"

# 5. Validate numerical data
for col in FEATURE_COLUMNS:
    ETTH1_DF[col] = pd.to_numeric(
        ETTH1_DF[col],
        errors="raise"
    )

assert not ETTH1_DF.isna().any().any(), (
    "Missing values detected"
)

values = ETTH1_DF[FEATURE_COLUMNS].to_numpy(
    dtype=np.float64
)

assert np.isfinite(values).all(), (
    "NaN or Inf detected"
)

# 6. Dataset fingerprint
DATA_SHA256 = hashlib.sha256(
    DATA_PATH.read_bytes()
).hexdigest()

# 7. Summary
print("=== ETTh1 DATA VALIDATION ===")

print("Dataset:", DATA_PATH)
print("Shape:", ETTH1_DF.shape)
print("Numerical channels:", len(FEATURE_COLUMNS))

print("\nStart:", ETTH1_DF["date"].iloc[0])
print("End:", ETTH1_DF["date"].iloc[-1])
print("Frequency: 1 hour")

print("\nMissing values:", int(ETTH1_DF.isna().sum().sum()))
print("Duplicate timestamps:", int(
    ETTH1_DF["date"].duplicated().sum()
))

print("\nSHA256:", DATA_SHA256)

print("\n=== FIRST 5 ROWS ===")
display(ETTH1_DF.head())

print("\n=== FEATURE STATISTICS ===")
display(
    ETTH1_DF[FEATURE_COLUMNS]
    .describe()
    .T
    .round(4)
)

print("\nETTh1 DATA VALIDATION PASSED!")


=== ETTh1 DATA VALIDATION ===
Dataset: c:\Users\Win 11 New\Desktop\pa gk\datasets\ETTh1\ETTh1.csv
Shape: (17420, 8)
Numerical channels: 7

Start: 2016-07-01 00:00:00
End: 2018-06-26 19:00:00
Frequency: 1 hour

Missing values: 0
Duplicate timestamps: 0

SHA256: f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066

=== FIRST 5 ROWS ===


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000



=== FEATURE STATISTICS ===


,count,mean,std,min,25%,50%,75%,max
HUFL,17420.0,7.3751,7.0677,-22.706,5.827,8.774,11.788,23.644
HULL,17420.0,2.2422,2.0423,-4.756,0.737,2.210,3.684,10.114
MUFL,17420.0,4.3002,6.8270,-25.088,3.296,5.970,8.635,17.341
MULL,17420.0,0.8816,1.8093,-5.934,-0.284,0.959,2.203,7.747
LUFL,17420.0,3.0661,1.1645,-1.188,2.315,2.833,3.625,8.498
LULL,17420.0,0.8569,0.5996,-1.371,0.670,0.975,1.218,3.046
OT,17420.0,13.3247,8.5669,-4.080,6.964,11.396,18.079,46.007



ETTh1 DATA VALIDATION PASSED!


ETTh1 SPLIT & STANDARDIZATION

In [3]:
import torch
from sklearn.preprocessing import StandardScaler

# 1. Experiment configuration
SEQ_LEN = 96
PRED_LENGTHS = [96, 192, 336, 720]
N_CHANNELS = len(FEATURE_COLUMNS)

# Official ETTh1 split boundaries
TRAIN_END = 12 * 30 * 24
VAL_END = TRAIN_END + 4 * 30 * 24
TEST_END = VAL_END + 4 * 30 * 24

assert (TRAIN_END, VAL_END, TEST_END) == (
    8640, 11520, 14400
)

assert len(ETTH1_DF) == 17420

# 2. Extract numerical features
raw_values = ETTH1_DF[
    FEATURE_COLUMNS
].to_numpy(dtype=np.float64)

# 3. Fit StandardScaler on TRAIN ONLY
SCALER = StandardScaler()

SCALER.fit(raw_values[:TRAIN_END])

scaled_values = SCALER.transform(
    raw_values
).astype(np.float32)

assert np.isfinite(scaled_values).all()

# 4. Dataset boundaries
# Validation and Test include SEQ_LEN past timestamps
# as input context, but not as target labels.

SPLIT_BORDERS = {
    "train": (0, TRAIN_END),
    "val": (TRAIN_END - SEQ_LEN, VAL_END),
    "test": (VAL_END - SEQ_LEN, TEST_END)
}

TARGET_STARTS = {
    "train": 0,
    "val": TRAIN_END,
    "test": VAL_END
}

ETTH1_SPLITS = {}
summary_rows = []

for split_name, (start, end) in SPLIT_BORDERS.items():

    series = scaled_values[start:end].copy()

    ETTH1_SPLITS[split_name] = torch.from_numpy(
        series
    )

    # Validation/Test labels must start in their own split
    target_start = TARGET_STARTS[split_name]

    if split_name != "train":
        assert start + SEQ_LEN == target_start

    row = {
        "Split": split_name,
        "Data_Start": start,
        "Data_End": end,
        "Total_Rows_With_Context": len(series),
        "Target_Start": target_start,
        "First_Timestamp": str(
            ETTH1_DF["date"].iloc[start]
        ),
        "Last_Timestamp": str(
            ETTH1_DF["date"].iloc[end - 1]
        )
    }

    for horizon in PRED_LENGTHS:
        n_windows = len(series) - SEQ_LEN - horizon + 1

        assert n_windows > 0

        row[f"Windows_H{horizon}"] = n_windows

    summary_rows.append(row)

split_summary = pd.DataFrame(summary_rows)

# 5. Verify Train standardization
train_scaled = scaled_values[:TRAIN_END]

np.testing.assert_allclose(
    train_scaled.mean(axis=0),
    np.zeros(N_CHANNELS),
    atol=1e-5
)

np.testing.assert_allclose(
    train_scaled.std(axis=0),
    np.ones(N_CHANNELS),
    atol=1e-5
)

# 6. Verify chronological boundaries
assert TRAIN_END <= VAL_END <= TEST_END
assert TEST_END <= len(ETTH1_DF)

assert ETTH1_SPLITS["train"].shape == (8640, 7)
assert ETTH1_SPLITS["val"].shape == (2976, 7)
assert ETTH1_SPLITS["test"].shape == (2976, 7)

# 7. Save reusable dataset bundle
PREPARED_PATH = RESULTS_DIR / "etth1_prepared.pt"

bundle = {
    "dataset": "ETTh1",
    "source_sha256": DATA_SHA256,
    "features": FEATURE_COLUMNS,
    "seq_len": SEQ_LEN,
    "pred_lengths": PRED_LENGTHS,
    "channels": N_CHANNELS,
    "normalization": "StandardScaler",
    "scaler_mean": torch.tensor(
        SCALER.mean_, dtype=torch.float64
    ),
    "scaler_scale": torch.tensor(
        SCALER.scale_, dtype=torch.float64
    ),
    "train": ETTH1_SPLITS["train"],
    "val": ETTH1_SPLITS["val"],
    "test": ETTH1_SPLITS["test"],
    "split_borders": SPLIT_BORDERS,
    "target_starts": TARGET_STARTS
}

torch.save(bundle, PREPARED_PATH)

split_summary.to_csv(
    RESULTS_DIR / "split_summary.csv",
    index=False
)

# 8. Reload verification
restored = torch.load(
    PREPARED_PATH,
    map_location="cpu",
    weights_only=True
)

assert restored["source_sha256"] == DATA_SHA256
assert restored["seq_len"] == 96
assert torch.equal(
    restored["train"],
    ETTH1_SPLITS["train"]
)

# 9. Print results
print("=== ETTh1 SPLIT & STANDARDIZATION ===")

display(split_summary)

print("\nTrain mean (scaled):",
      np.round(train_scaled.mean(axis=0), 6))

print("Train std (scaled):",
      np.round(train_scaled.std(axis=0), 6))

print("\nSequence length:", SEQ_LEN)
print("Forecast horizons:", PRED_LENGTHS)
print("Channels:", N_CHANNELS)

print("\nSaved:", PREPARED_PATH)
print("Saved:", RESULTS_DIR / "split_summary.csv")

print("\nETTh1 SPLIT & STANDARDIZATION PASSED!")


=== ETTh1 SPLIT & STANDARDIZATION ===


c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


,Split,Data_Start,Data_End,Total_Rows_With_Context,Target_Start,First_Timestamp,Last_Timestamp,Windows_H96,Windows_H192,Windows_H336,Windows_H720
0,train,0,8640,8640,0,2016-07-01 00:00:00,2017-06-25 23:00:00,8449,8353,8209,7825
1,val,8544,11520,2976,8640,2017-06-22 00:00:00,2017-10-23 23:00:00,2785,2689,2545,2161
2,test,11424,14400,2976,11520,2017-10-20 00:00:00,2018-02-20 23:00:00,2785,2689,2545,2161



Train mean (scaled): [-0. -0. -0.  0.  0.  0.  0.]
Train std (scaled): [1. 1. 1. 1. 1. 1. 1.]

Sequence length: 96
Forecast horizons: [96, 192, 336, 720]
Channels: 7

Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\etth1_data\etth1_prepared.pt
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\etth1_data\split_summary.csv

ETTh1 SPLIT & STANDARDIZATION PASSED!


ETTh1 SLIDING WINDOWS VALIDATION

In [4]:
from torch.utils.data import Dataset, DataLoader

# 1. Sliding Window Dataset
class ETTh1WindowDataset(Dataset):

    def __init__(self, series, seq_len, pred_len):
        self.series = series
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)

        assert self.series.ndim == 2
        assert len(self.series) >= self.seq_len + self.pred_len

    def __len__(self):
        return (
            len(self.series)
            - self.seq_len
            - self.pred_len
            + 1
        )

    def __getitem__(self, idx):
        if not 0 <= idx < len(self):
            raise IndexError(idx)

        s = idx
        e = s + self.seq_len
        future_end = e + self.pred_len

        X = self.series[s:e]
        Y = self.series[e:future_end]

        return X, Y


# 2. Create datasets for all horizons
WINDOW_DATASETS = {}
window_rows = []

LABEL_BOUNDARIES = {
    "train": (0, TRAIN_END),
    "val": (TRAIN_END, VAL_END),
    "test": (VAL_END, TEST_END)
}

for horizon in PRED_LENGTHS:

    for split_name, series in ETTH1_SPLITS.items():

        ds = ETTh1WindowDataset(
            series=series,
            seq_len=SEQ_LEN,
            pred_len=horizon
        )

        WINDOW_DATASETS[(split_name, horizon)] = ds

        # First and last windows
        X_first, Y_first = ds[0]
        X_last, Y_last = ds[len(ds) - 1]

        assert X_first.shape == (SEQ_LEN, N_CHANNELS)
        assert Y_first.shape == (horizon, N_CHANNELS)

        assert X_last.shape == (SEQ_LEN, N_CHANNELS)
        assert Y_last.shape == (horizon, N_CHANNELS)

        assert torch.isfinite(X_first).all()
        assert torch.isfinite(Y_first).all()
        assert torch.isfinite(X_last).all()
        assert torch.isfinite(Y_last).all()

        # 3. Absolute indices in original ETTh1
        split_start, split_end = SPLIT_BORDERS[split_name]

        first_target_idx = split_start + SEQ_LEN
        last_target_idx = split_end - 1

        allowed_start, allowed_end = (
            LABEL_BOUNDARIES[split_name]
        )

        # All prediction targets must belong to their split
        assert first_target_idx >= allowed_start
        assert last_target_idx < allowed_end

        # Confirm target indices for first and last windows
        last_window_start = len(ds) - 1

        assert (
            split_start
            + last_window_start
            + SEQ_LEN
            + horizon
        ) == split_end

        # 4. Verify exact values against original scaled data
        expected_X_first = torch.from_numpy(
            scaled_values[
                split_start : split_start + SEQ_LEN
            ]
        )

        expected_Y_first = torch.from_numpy(
            scaled_values[
                first_target_idx :
                first_target_idx + horizon
            ]
        )

        expected_Y_last = torch.from_numpy(
            scaled_values[
                split_end - horizon : split_end
            ]
        )

        assert torch.equal(X_first, expected_X_first)
        assert torch.equal(Y_first, expected_Y_first)
        assert torch.equal(Y_last, expected_Y_last)

        # Input must precede the forecast target
        assert (
            ETTH1_DF["date"].iloc[first_target_idx]
            - ETTH1_DF["date"].iloc[first_target_idx - 1]
        ) == pd.Timedelta(hours=1)

        # 5. Record summary
        window_rows.append({
            "Split": split_name,
            "Horizon": horizon,
            "Windows": len(ds),
            "X_Shape": str(tuple(X_first.shape)),
            "Y_Shape": str(tuple(Y_first.shape)),
            "First_Target": str(
                ETTH1_DF["date"].iloc[first_target_idx]
            ),
            "Last_Target": str(
                ETTH1_DF["date"].iloc[last_target_idx]
            ),
            "Status": "PASSED"
        })


# 6. Validate PyTorch DataLoader
train_loader = DataLoader(
    WINDOW_DATASETS[("train", 96)],
    batch_size=32,
    shuffle=False,
    num_workers=0
)

X_batch, Y_batch = next(iter(train_loader))

assert X_batch.shape == (32, SEQ_LEN, N_CHANNELS)
assert Y_batch.shape == (32, 96, N_CHANNELS)

# 7. Save validation summary
WINDOW_SUMMARY = pd.DataFrame(window_rows)

WINDOW_SUMMARY.to_csv(
    RESULTS_DIR / "window_summary.csv",
    index=False
)

assert len(WINDOW_DATASETS) == 12
assert len(WINDOW_SUMMARY) == 12
assert (WINDOW_SUMMARY["Status"] == "PASSED").all()

# 8. Display results
print("=== ETTh1 SLIDING WINDOWS VALIDATION ===")

display(WINDOW_SUMMARY)

print("\nDataLoader check:")
print("X_batch:", tuple(X_batch.shape))
print("Y_batch:", tuple(Y_batch.shape))

print("\nDataset configurations:", len(WINDOW_DATASETS))
print("Saved:", RESULTS_DIR / "window_summary.csv")

print("\nNo future-label leakage detected.")
print("ETTh1 SLIDING WINDOWS VALIDATION PASSED!")


=== ETTh1 SLIDING WINDOWS VALIDATION ===


,Split,Horizon,Windows,X_Shape,Y_Shape,First_Target,Last_Target,Status
0,train,96,8449,"(96, 7)","(96, 7)",2016-07-05 00:00:00,2017-06-25 23:00:00,PASSED
1,val,96,2785,"(96, 7)","(96, 7)",2017-06-26 00:00:00,2017-10-23 23:00:00,PASSED
2,test,96,2785,"(96, 7)","(96, 7)",2017-10-24 00:00:00,2018-02-20 23:00:00,PASSED
3,train,192,8353,"(96, 7)","(192, 7)",2016-07-05 00:00:00,2017-06-25 23:00:00,PASSED
4,val,192,2689,"(96, 7)","(192, 7)",2017-06-26 00:00:00,2017-10-23 23:00:00,PASSED
5,test,192,2689,"(96, 7)","(192, 7)",2017-10-24 00:00:00,2018-02-20 23:00:00,PASSED
6,train,336,8209,"(96, 7)","(336, 7)",2016-07-05 00:00:00,2017-06-25 23:00:00,PASSED
7,val,336,2545,"(96, 7)","(336, 7)",2017-06-26 00:00:00,2017-10-23 23:00:00,PASSED
8,test,336,2545,"(96, 7)","(336, 7)",2017-10-24 00:00:00,2018-02-20 23:00:00,PASSED
9,train,720,7825,"(96, 7)","(720, 7)",2016-07-05 00:00:00,2017-06-25 23:00:00,PASSED



DataLoader check:
X_batch: (32, 96, 7)
Y_batch: (32, 96, 7)

Dataset configurations: 12
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\etth1_data\window_summary.csv

No future-label leakage detected.
ETTh1 SLIDING WINDOWS VALIDATION PASSED!
